In [0]:
from pyspark.sql import functions as F
def validate_current_state(df):
    """
    Validates the df before loading to the current state table.
    """
    if df.filter((F.col("domain").isNull()) | (F.col("category").isNull())).limit(1).count():
        raise ValueError("Current state contains a null domain or category")

    if (
        df.groupBy("domain").count()
        .filter(F.col("count") > 1)
        .limit(1).count()
    ):
        raise ValueError("Current state contains duplicate domains")

    # Prevent accidental closure of the entire history.
    if df.limit(1).count() == 0:
        raise ValueError("Current state is empty")

    print("Validated current state. No errors found")

In [0]:
def validate_parsed_count(
    parsed_df,
    run_id,
    run_date,
    audit_table,
    max_drop_ratio=0.70,
    expected_feeds=["feed_a", "feed_b", "feed_c", "feed_d"],
):
    """
    Group the input DataFrame by feed_name and validate each feed.

    max_drop_ratio=0.70 allows a record count decrease of up to 70%.
    expected_feeds optionally includes feeds missing from the DataFrame.
    """

    if not 0 <= max_drop_ratio <= 1:
        raise ValueError("max_drop_ratio must be between 0 and 1")

    # Count records for each feed before applying any filters.
    counts_by_feed = {
        row["feed_name"]: row["count"]
        for row in parsed_df.groupBy("feed_name").count().collect()
    }

    if None in counts_by_feed:
        raise ValueError("DataFrame contains records with a null feed_name")

    # Include expected feeds even if they contain no parsed records.
    feeds_to_validate = set(counts_by_feed)

    if expected_feeds is not None:
        feeds_to_validate.update(expected_feeds)

    if not feeds_to_validate:
        raise ValueError(
            "DataFrame is empty and no expected feeds were provided"
        )

    accepted_by_feed = {
        row["feed_name"]: row["count"]
        for row in parsed_df.filter(F.col("validation_status") == "accepted domain")
        .select("feed_name", "domain").distinct().groupBy("feed_name").count().collect()
    }
    validation_errors = []

    for feed_name in sorted(feeds_to_validate):
        current_count = counts_by_feed.get(feed_name, 0)

        # Retrieve the previous accepted count for this specific feed.
        previous_count = get_previous_parsed_count(
            audit_table=audit_table,
            feed_name=feed_name,
            run_id=run_id,
        )

        drop_ratio = None
        errors = []

        # Validation 1.
        # check that the feed contains at least one parsed record.
        if current_count == 0:
            errors.append("Parsed feed is empty or missing")

        # Validation 2.
        # compare against the previous accepted count for the same feed
        if previous_count is not None and previous_count > 0:
            drop_ratio = (
                previous_count - current_count
            ) / previous_count

            if drop_ratio > max_drop_ratio:
                errors.append(
                    f"Record count dropped by {drop_ratio:.2%}; "
                    f"maximum allowed drop is {max_drop_ratio:.2%}. "
                    f"Previous: {previous_count}, current: {current_count}"
                )

        # A feed full of IPs or invalid records must not pass just because it is large.
        accepted_count = accepted_by_feed.get(feed_name, 0)
        previous_accepted = get_previous_parsed_count(
            audit_table, feed_name, run_id, stage="accepted domains"
        )
        accepted_drop = ((previous_accepted - accepted_count) / previous_accepted
                         if previous_accepted else None)
        accepted_failed = accepted_count == 0 or (
            accepted_drop is not None and accepted_drop > max_drop_ratio
        )
        if accepted_failed:
            errors.append("Accepted unique domains are empty or dropped beyond the threshold")
        log_record_count(
            audit_table, run_id, run_date, feed_name, "accepted domains",
            accepted_count, "failed" if accepted_failed else "logged",
            previous_accepted, accepted_drop,
        )
        status = "failed" if errors else "logged"

        # each feed independently, including failed validations.
        log_record_count(
            audit_table=audit_table,
            run_id=run_id,
            run_date=run_date,
            feed_name=feed_name,
            stage="parsed",
            record_count=current_count,
            previous_count=previous_count,
            ratio=drop_ratio,
            status=status,
        )

        counts_by_feed[feed_name] = current_count

        if errors:
            validation_errors.append(f"{feed_name}: " + ", ".join(errors))

    # raise only after all feeds have been validated and saved to audit_table
    if validation_errors:
        raise ValueError(
            "Feed validation failed:\n" + "\n".join(validation_errors)
        )

    print("Feed validation passed. Parsed feeds contain following counts:")

    return counts_by_feed

In [0]:
def validate_count_balance(audit_table, run_id, output_count):
    """Validate feed statuses and reconcile input, filtered and output counts."""
     # 1. Check the latest validation status for each feed and stage.
    checks = spark.sql(f"""
        WITH latest_records AS (
            SELECT
                feed_name,
                stage,
                status,
                record_count,
                ROW_NUMBER() OVER (
                    PARTITION BY feed_name, stage
                    ORDER BY logged_at DESC
                ) AS rn
            FROM {audit_table}
            WHERE run_id = '{run_id}'
              AND stage IN ('parsed', 'accepted domains')
        )
        SELECT feed_name, stage, status, record_count
        FROM latest_records
        WHERE rn = 1
    """).collect()

    expected_feeds = {"feed_a", "feed_b", "feed_c", "feed_d"}
    actual_feeds = {row.feed_name for row in checks}

    # Four feeds, each with two checks.
    if actual_feeds != expected_feeds or len(checks) != 8:
        raise ValueError("Missing feed validation results.")

    for row in checks:
        if row.status != "passed":
            raise ValueError(
                f"Validation not passed: {row.feed_name}, "
                f"stage={row.stage}, status={row.status}, "
                f"count={row.record_count}"
            )

    # 2. Reconcile original file lines with parsed records for each feed.
    parser_counts = spark.sql(f"""
        WITH latest_records AS (
            SELECT
                feed_name,
                stage,
                record_count,
                ROW_NUMBER() OVER (
                    PARTITION BY feed_name, stage
                    ORDER BY logged_at DESC
                ) AS rn
            FROM {audit_table}
            WHERE run_id = '{run_id}'
              AND stage IN (
                  'input lines',
                  'skipped lines',
                  'extra aliases',
                  'system hosts',
                  'parsed'
              )
        )
        SELECT
            feed_name,
            MAX(CASE WHEN stage = 'input lines'
                THEN record_count END) AS input_lines,
            MAX(CASE WHEN stage = 'skipped lines'
                THEN record_count END) AS skipped_lines,
            MAX(CASE WHEN stage = 'extra aliases'
                THEN record_count END) AS extra_aliases,
            MAX(CASE WHEN stage = 'system hosts'
                THEN record_count END) AS system_hosts,
            MAX(CASE WHEN stage = 'parsed'
                THEN record_count END) AS parsed_count
        FROM latest_records
        WHERE rn = 1
        GROUP BY feed_name
        ORDER BY feed_name
    """).collect()

    input_count = 0

    for row in parser_counts:
        if None in (
            row.input_lines,
            row.skipped_lines,
            row.extra_aliases,
            row.system_hosts,
            row.parsed_count,
        ):
            raise ValueError(
                f"Missing parser counts for {row.feed_name}."
            )

        expected_parsed = (
            row.input_lines
            - row.skipped_lines
            + row.extra_aliases
            - row.system_hosts
        )

        print(
            f"{row.feed_name}: "
            f"{row.input_lines:,} input lines "
            f"- {row.skipped_lines:,} skipped lines "
            f"+ {row.extra_aliases:,} extra aliases "
            f"- {row.system_hosts:,} system hosts "
            f"= {expected_parsed:,} expected parsed records "
            f"(actual: {row.parsed_count:,})."
        )

        if expected_parsed != row.parsed_count:
            raise ValueError(
                f"Parser count mismatch for {row.feed_name}: "
                f"expected={expected_parsed}, actual={row.parsed_count}"
            )

        input_count += row.parsed_count

    print(f"Run {run_id} has {input_count:,} parsed records.")

    # 3. Count records removed after parsing, including deduplication.
    filtered_out_rows = spark.sql(f"""
        WITH latest_records AS (
            SELECT
                feed_name,
                status,
                record_count,
                ROW_NUMBER() OVER (
                    PARTITION BY feed_name, status
                    ORDER BY logged_at DESC
                ) AS rn
            FROM {audit_table}
            WHERE run_id = '{run_id}'
              AND stage = 'filtered out'
        )
        SELECT feed_name, status, record_count
        FROM latest_records
        WHERE rn = 1
        ORDER BY feed_name, status
    """).collect()

    filtered_out_count = 0

    for row in filtered_out_rows:
        print(
            f"Removed {row.record_count:,} records: "
            f"{row.status} ({row.feed_name})."
        )
        filtered_out_count += row.record_count

    print(f"Total removed records: {filtered_out_count:,}")
    print(f"Output records: {output_count:,}")

    # 4. Validate the final balance
    difference = input_count - filtered_out_count - output_count

    if difference != 0:
        raise ValueError(
            f"Count mismatch: "
            f"{input_count:,} parsed "
            f"- {filtered_out_count:,} removed "
            f"- {output_count:,} output "
            f"= {difference:,}."
        )

    print(
        f"Count balance passed: "
        f"{input_count:,} - {filtered_out_count:,} = {output_count:,}."
    )